<a href="https://colab.research.google.com/github/LRManamperi/Admin-Dashboard/blob/main/tinyMLHeadpose_pruning_quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


You can download the dataset used in this notebook from the following Google Drive link:

[Headpose Dataset](https://drive.google.com/drive/folders/1r0OHH82z2qv49zioYuk50Y85xhdPrvRF?usp=sharing)

Get a copy of this Dir to your Google Drive to get started.

#Installing Dependencies

In [ ]:

!pip uninstall -y keras tensorflow tensorflow-model-optimization
!pip install tensorflow==2.12 tensorflow-model-optimization

Found existing installation: keras 2.12.0
Uninstalling keras-2.12.0:
  Successfully uninstalled keras-2.12.0
Found existing installation: tensorflow 2.12.0
Uninstalling tensorflow-2.12.0:
  Successfully uninstalled tensorflow-2.12.0
Found existing installation: tensorflow-model-optimization 0.8.0
Uninstalling tensorflow-model-optimization-0.8.0:
  Successfully uninstalled tensorflow-model-optimization-0.8.0
  Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached tensorflow_model_optimization-0.8.0-py2.py3-none-any.whl.metadata (904 bytes)
  Using cached keras-2.12.0-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached tensorflow-2.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (586.0 MB)
Using cached tensorflow_model_optimization-0.8.0-py2.py3-none-any.whl (242 kB)
Using cached keras-2.12.0-py2.py3-none-any.whl (1.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packa

In [ ]:
import os
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
# Define the base path, data path, and BIWI dataset path

base_path='/content/drive/MyDrive/headposeData/'
data_path=os.path.join(base_path,'data')
biwi_path=os.path.join(data_path,'biwi')


#dataset preperation

In [ ]:
all_files = os.listdir(biwi_path)
print(all_files)
# List all files in the specified directory

['BIWI_train.npz', 'BIWI_test.npz']


In [ ]:
npz_files = []
for file in all_files:
    if file.endswith('.npz'):
        npz_files.append(file)
print(npz_files)

['BIWI_train.npz', 'BIWI_test.npz']


In [ ]:
loaded_data = {}
for file in npz_files:
    file_path = os.path.join(biwi_path, file)
    data = np.load(file_path)
    key = file.replace('.npz', '')
    loaded_data[key] = data

In [ ]:
train_data=loaded_data['BIWI_train']
test_data=loaded_data['BIWI_test']


In [ ]:
train_images = train_data['image']
train_poses = train_data['pose']
test_images = test_data['image']
test_poses = test_data['pose']

print("Shape of training images:", train_images.shape)
print("Shape of training poses:", train_poses.shape)
print("Shape of testing images:", test_images.shape)
print("Shape of testing poses:", test_poses.shape)

Shape of training images: (10613, 64, 64, 3)
Shape of training poses: (10613, 3)
Shape of testing images: (5065, 64, 64, 3)
Shape of testing poses: (5065, 3)


Now I'll split the training data into training and validation sets using `train_test_split`.

In [ ]:

# Split the training data into training and validation sets
train_images, val_images, train_poses, val_poses = train_test_split(
    train_images, train_poses, test_size=0.2, random_state=42
)

print("Shape of training images after split:", train_images.shape)
print("Shape of validation images after split:", val_images.shape)
print("Shape of training poses after split:", train_poses.shape)
print("Shape of validation poses after split:", val_poses.shape)

Shape of training images after split: (8490, 64, 64, 3)
Shape of validation images after split: (2123, 64, 64, 3)
Shape of training poses after split: (8490, 3)
Shape of validation poses after split: (2123, 3)


Convert all the images (training, validation, and testing) to grayscale.
### 💡 Why Grayscale Reduces the Number of Parameters

In a Convolutional Neural Network (CNN), the **number of parameters** in the first convolutional layer depends on:

\[
\text{Parameters} = (\text{Kernel Height} \times \text{Kernel Width} \times \text{Input Channels} \times \text{Output Channels}) + \text{Output Channels (biases)}
\]

**Example:**
- Kernel size: \(3 \times 3\)
- Output channels: \(32\)
- Input channels:
  - **RGB** → 3 channels
  - **Grayscale** → 1 channel

**RGB case**:
\[
3 \times 3 \times 3 \times 32 = 864 \text{ weights}
\]

**Grayscale case**:
\[
3 \times 3 \times 1 \times 32 = 288 \text{ weights}
\]

**Result:** Converting to grayscale **reduces parameters in the first convolutional layer by 66%**.

**Benefits:**
- **Lower memory usage** (smaller tensors)
- **Faster training/inference** (less computation in early layers)
- **Less risk of overfitting** when color is not important for the task (e.g., head pose estimation)


In [ ]:
# Convert images to grayscale
train_images_gray = np.dot(train_images[...,:3], [0.2989, 0.5870, 0.1140])
val_images_gray = np.dot(val_images[...,:3], [0.2989, 0.5870, 0.1140])
test_images_gray = np.dot(test_images[...,:3], [0.2989, 0.5870, 0.1140])

# Add a channel dimension for grayscale
train_images_gray = np.expand_dims(train_images_gray, axis=-1)
val_images_gray = np.expand_dims(val_images_gray, axis=-1)
test_images_gray = np.expand_dims(test_images_gray, axis=-1)

print("Shape of grayscale training images:", train_images_gray.shape)
print("Shape of grayscale validation images:", val_images_gray.shape)
print("Shape of grayscale testing images:", test_images_gray.shape)

Shape of grayscale training images: (8490, 64, 64, 1)
Shape of grayscale validation images: (2123, 64, 64, 1)
Shape of grayscale testing images: (5065, 64, 64, 1)


Now, I'll set up an `ImageDataGenerator` for the training set with zoom augmentation.

In [ ]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an ImageDataGenerator for training with zoom augmentation
train_datagen = ImageDataGenerator(
    zoom_range=0.2,  # Apply random zoom
    rescale=1./255   # Normalize pixel values
)

# Create a separate generator for validation and testing (only for normalization)
# No augmentation is applied to validation and test sets
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create the generators for the datasets
train_generator = train_datagen.flow(
    train_images_gray,
    train_poses,
    batch_size=32,
    shuffle=True
)

val_generator = val_test_datagen.flow(
    val_images_gray,
    val_poses,
    batch_size=32,
    shuffle=False
)

test_generator = val_test_datagen.flow(
    test_images_gray,
    test_poses,
    batch_size=32,
    shuffle=False
)

print("Data generators created.")

Data generators created.


# **model preperation**

### 📏 Custom Mean Absolute Error (MAE) Metric for Head Pose Estimation

In regression tasks like **head pose estimation**, the **Mean Absolute Error (MAE)** is a common metric to evaluate model performance.  

**Definition:**
\[
MAE = \frac{1}{N} \sum_{i=1}^{N} \left| y_{\text{pred}, i} - y_{\text{true}, i} \right|
\]
where:
- \( y_{\text{pred}} \) = predicted value
- \( y_{\text{true}} \) = ground truth value
- \( N \) = number of samples

**How it works in this code:**
- **`K.abs()`** computes the absolute difference between predictions and true values.
- **`K.mean(..., axis=0)`** calculates the average error **per output dimension** (e.g., yaw, pitch, roll).
- This is useful because it shows how well the model performs **for each individual angle** rather than combining them into a single number.

**Why use MAE here?**
- Angular errors are **interpretable** in degrees or radians.
- Less sensitive to outliers compared to metrics like MSE.
- Allows tracking of pose-specific accuracy during training.


In [ ]:

from tensorflow.keras import backend

def mae(y_true, y_pred):
	# return backend.sqrt(backend.mean(backend.square(y_pred - y_true), axis=0))
	return backend.mean(backend.abs(y_pred - y_true), axis=0)
  # return np.mean(np.abs(y_true - y_pred), axis = 0)

### Simplified MobileNetV1 for Head Pose Estimation (Grayscale Input)

This code builds a **lightweight CNN** inspired by **MobileNetV1**, optimized for low-power and embedded inference.  

#### **Key Features:**
1. **Grayscale Input**  
   - Input shape: `(64, 64, 1)`  
   - Using grayscale reduces the first convolution’s parameters by ~66% compared to RGB.

2. **Depthwise Separable Convolutions**  
   - MobileNetV1 replaces standard convolutions with a **depthwise convolution** (per-channel filtering) followed by a **pointwise convolution** (1×1) to combine channels.  
   - Greatly reduces parameters and computation (FLOPs) without large accuracy loss.

3. **Architecture Overview:**
   - **Initial Conv Layer:** Standard 3×3 convolution to extract low-level features.
   - **Depthwise Separable Blocks:** Each block has:
     - Depthwise 3×3 conv → BatchNorm → ReLU
     - Pointwise 1×1 conv → BatchNorm → ReLU
   - **Stride=2 blocks** reduce spatial resolution for downsampling.
   - **Channel expansion** from 32 → 1024 as depth increases.

4. **Global Average Pooling (GAP)**  
   - Reduces the spatial dimensions to a single feature vector.
   - More parameter-efficient than flattening.

5. **Dense Layers for Regression**  
   - Fully connected layers refine features for predicting continuous outputs.

6. **Final Output Layer**  
   - **3 neurons**, linear activation → yaw, pitch, roll angles for head pose estimation.

#### **Why MobileNetV1 for TinyML?**
- Extremely efficient for **microcontrollers** and **mobile devices**.
- Depthwise separable convs cut computation by ~8–9× compared to standard convs.
- Well-suited for **real-time** pose estimation on edge devices.

**Output:**  
The model predicts **3 continuous values** representing yaw, pitch, and roll.


In [ ]:
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D, BatchNormalization, ReLU, GlobalAveragePooling2D, Dense, ZeroPadding2D
from tensorflow.keras.models import Model

# Define the input layer
input_shape = (64, 64, 1) # Grayscale images
input_tensor = Input(shape=input_shape)

# MobileNetV1 base (simplified manual construction)
# This is a simplified representation and not the full MobileNetV1 architecture
# It includes an initial conv layer and a few depthwise separable blocks

def _depthwise_separable_conv(inputs, pointwise_conv_filters, alpha,
                              depth_multiplier=1, strides=(1, 1), block_id=1):
    channel_axis = -1

    if strides == (1, 1):
        x = inputs
    else:
        x = ZeroPadding2D(((0, 1), (0, 1)), name='conv_pad_%d' % block_id)(inputs)
    x = DepthwiseConv2D((3, 3),
                        padding='same' if strides == (1, 1) else 'valid',
                        depth_multiplier=depth_multiplier,
                        strides=strides,
                        use_bias=False,
                        name='conv_dw_%d' % block_id)(x)
    x = BatchNormalization(axis=channel_axis, name='conv_dw_%d_bn' % block_id)(x)
    x = ReLU(6., name='conv_dw_%d_relu' % block_id)(x)

    x = Conv2D(pointwise_conv_filters, (1, 1),
               padding='same',
               use_bias=False,
               strides=(1, 1),
               name='conv_pw_%d' % block_id)(x)
    x = BatchNormalization(axis=channel_axis, name='conv_pw_%d_bn' % block_id)(x)
    x = ReLU(6., name='conv_pw_%d_relu' % block_id)(x)
    return x

# Initial convolution layer
x = Conv2D(32, (3, 3), padding='same', use_bias=False, strides=(2, 2), name='conv1')(input_tensor)
x = BatchNormalization(axis=-1, name='conv1_bn')(x)
x = ReLU(6., name='conv1_relu')(x)

# Depthwise separable convolution blocks
x = _depthwise_separable_conv(x, 64, 1.0, block_id=1)
x = _depthwise_separable_conv(x, 128, 1.0, strides=(2, 2), block_id=2)
x = _depthwise_separable_conv(x, 128, 1.0, block_id=3)
x = _depthwise_separable_conv(x, 256, 1.0, strides=(2, 2), block_id=4)
x = _depthwise_separable_conv(x, 256, 1.0, block_id=5)
x = _depthwise_separable_conv(x, 512, 1.0, strides=(2, 2), block_id=6)
x = _depthwise_separable_conv(x, 512, 1.0, block_id=7)
x = _depthwise_separable_conv(x, 512, 1.0, block_id=8)
x = _depthwise_separable_conv(x, 512, 1.0, block_id=9)
x = _depthwise_separable_conv(x, 512, 1.0, block_id=10)
x = _depthwise_separable_conv(x, 512, 1.0, block_id=11)
x = _depthwise_separable_conv(x, 1024, 1.0, strides=(2, 2), block_id=12)
x = _depthwise_separable_conv(x, 1024, 1.0, block_id=13)


# Global Average Pooling layer
x = GlobalAveragePooling2D()(x)

# Dense layers
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)

# Final output layer
output_tensor = Dense(3, activation='linear', name='predictions')(x)

# Create the model
model = Model(inputs=input_tensor, outputs=output_tensor)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 64, 64, 1)]       0         
                                                                 
 conv1 (Conv2D)              (None, 32, 32, 32)        288       
                                                                 
 conv1_bn (BatchNormalizatio  (None, 32, 32, 32)       128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 32, 32, 32)        0         
                                                                 
 conv_dw_1 (DepthwiseConv2D)  (None, 32, 32, 32)       288       
                                                                 
 conv_dw_1_bn (BatchNormaliz  (None, 32, 32, 32)       128       
 ation)                                                      

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend
import tensorflow as tf

def mae(y_true, y_pred):
        # return backend.sqrt(backend.mean(backend.square(y_pred - y_true), axis=0))
        y_true = tf.cast(y_true, y_pred.dtype) # Cast y_true to the dtype of y_pred
        return backend.mean(backend.abs(y_pred - y_true), axis=0)
  # return np.mean(np.abs(y_true - y_pred), axis = 0)


# Compile the model
model.compile(optimizer=Adam(learning_rate=0.01),
              loss='mean_squared_error',
              metrics=[mae, 'mean_absolute_error'])

print("Model compiled successfully.")

Model compiled successfully.


# Prepare for training



In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os

# Create a ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(base_path, 'best_model_weights.h5'),  # Path to save the model file
    save_weights_only=True,            # Save only the weights
    monitor='val_loss',                # Metric to monitor
    save_best_only=True,               # Save only the best model
    verbose=1                          # Log when a model is saved
)

# Create an EarlyStopping callback
early_stopping_callback = EarlyStopping(
    monitor='val_loss',  # Metric to monitor
    patience=10,         # Number of epochs with no improvement after which training will be stopped
    verbose=1            # Log when training is stopped
)

# Create a ReduceLROnPlateau callback
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # Metric to monitor
    factor=0.1,         # Factor by which the learning rate will be reduced. new_lr = lr * factor
    patience=5,         # Number of epochs with no improvement after which learning rate will be reduced.
    verbose=1           # Log when learning rate is reduced
)


# Define the number of training epochs
epochs = 50

print("Callbacks and training parameters defined.")

Callbacks and training parameters defined.


In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


Now I will train the model using the data generators and callbacks.

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_images_gray) // train_generator.batch_size,
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=len(val_images_gray) // val_generator.batch_size,
    callbacks=[checkpoint_callback, early_stopping_callback,reduce_lr]
)

print("Model training finished.")

Epoch 1/50
265/265 [==============================] - ETA: 0s - loss: 281.5083 - mae: 12.0062 - mean_absolute_error: 12.0019
Epoch 1: val_loss improved from inf to 898.18488, saving model to /content/drive/MyDrive/headposeData/best_model_weights.h5
265/265 [==============================] - 210s 758ms/step - loss: 281.5083 - mae: 12.0062 - mean_absolute_error: 12.0019 - val_loss: 898.1849 - val_mae: 24.7625 - val_mean_absolute_error: 24.7625 - lr: 0.0100
Epoch 2/50
123/265 [============>.................] - ETA: 1:41 - loss: 96.8155 - mae: 7.2771 - mean_absolute_error: 7.2771

KeyboardInterrupt: 

Now I'll evaluate the trained model on the test set.

In [ ]:
# Load the best saved weights
model.load_weights(os.path.join(base_path, 'best_model_weights.h5'))

print("Best model weights loaded.")

Best model weights loaded.


In [ ]:
# Evaluate the model on the test set
test_results = model.evaluate(
    test_generator,
    steps=len(test_images_gray) // test_generator.batch_size,
    verbose=1
)

# Print the test results
print("\nTest Loss:", test_results[0])
print("Test MAE:", test_results[1])
print("Test Mean Absolute Error:", test_results[2])

158/158 [==============================] - 12s 77ms/step - loss: 99.1120 - mae: 6.7882 - mean_absolute_error: 6.7882

Test Loss: 99.1119613647461
Test MAE: 6.788248062133789
Test Mean Absolute Error: 6.788247585296631


#Apply model Pruning

In [ ]:
import tensorflow_model_optimization

## Model Pruning with TensorFlow Model Optimization Toolkit (TFMOT)

This section applies **magnitude-based weight pruning** to reduce the size and computation cost of the MobileNetV1-based model.

---

#### **What is Pruning?**
Pruning removes less important weights (closer to zero) from the network, creating **sparse weight matrices**.  
- **Goal:** Reduce model size, speed up inference, and save energy — ideal for TinyML deployment.
- **Approach used:** **Magnitude-based pruning** → weights with smallest absolute values are set to zero.

---

#### **Pruning Schedule Parameters:**
- **`initial_sparsity=0.0`** → start with no weights pruned.
- **`final_sparsity=0.5`** → end with **50% of eligible weights pruned**.
- **`begin_step=0` & `end_step=20`** → pruning happens gradually over 20 training steps.
- **`PolynomialDecay`** → smoothly increases sparsity over time.

---

#### **Layer Selection:**
The `apply_pruning_to_layers()` function:
- **Prunes:**  
  - `Conv2D` layers  
  - `DepthwiseConv2D` layers  
  - `Dense` layers
- **Skips:**  
  - The **final output layer** (`predictions`) to avoid degrading output accuracy.
- Wraps eligible layers with `prune_low_magnitude()` from TFMOT.

---

#### **Workflow:**
1. **Clone model** → apply pruning wrapper only to selected layers.
2. **Recompile** → same optimizer, loss, and metrics as original model.
3. **Train pruned model** → during training, small weights are set to zero according to the schedule.

---

#### **Why Prune for TinyML?**
- Smaller model → **reduced flash storage** needs.
- Sparse weights → **faster inference** on some hardware.
- Energy-efficient → important for **battery-powered edge devices**.
- Can be combined with **quantization** for even greater compression.


In [ ]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.sparsity.keras import prune_low_magnitude, PolynomialDecay

# Define pruning parameters
num_images = len(train_images_gray)
batch_size = train_generator.batch_size
begin_step = 0
end_step = 3
initial_sparsity = 0.0
final_sparsity = 0.75

pruning_params = {
    'pruning_schedule': PolynomialDecay(initial_sparsity=initial_sparsity,
                                                             final_sparsity=final_sparsity,
                                                             begin_step=begin_step,
                                                             end_step=end_step)
}
# # 1) Set a fixed sparsity target
# pruning_params = {
#     "pruning_schedule": tfmot.sparsity.keras.ConstantSparsity(
#         target_sparsity=0.5,   # 50% zeros, pick your target
#         begin_step=0,
#         end_step=-1,           # keep it constant
#         frequency=100          # how often to update masks
#     )
# }
# Define a function to apply pruning to eligible layers
def apply_pruning_to_layers(layer):
    # You can customize this function to prune specific layer types or layers by name
    if isinstance(layer, tf.keras.layers.Conv2D) or isinstance(layer, tf.keras.layers.DepthwiseConv2D) or isinstance(layer, tf.keras.layers.Dense):
        # Don't prune the final output layer
        if layer.name == 'predictions':
            return layer
        # Apply the pruning wrapper to the layer
        return prune_low_magnitude(layer, **pruning_params)
    return layer

# Apply pruning to the model
model_pruning = tf.keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_layers,
)

# Recompile the pruned model
# Use the same optimizer, loss, and metrics as the original model
model_pruning.compile(optimizer=model.optimizer, # Use the optimizer from the original model
                      loss=model.loss,         # Use the loss from the original model
                      metrics=['mean_absolute_error'])     # Use only the built-in MAE metric

model_pruning.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 64, 64, 1)]       0         
                                                                 
 prune_low_magnitude_conv1 (  (None, 32, 32, 32)       578       
 PruneLowMagnitude)                                              
                                                                 
 conv1_bn (BatchNormalizatio  (None, 32, 32, 32)       128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 32, 32, 32)        0         
                                                                 
 prune_low_magnitude_conv_dw  (None, 32, 32, 32)       289       
 _1 (PruneLowMagnitude)                                          
                                                             

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
import os

# Create a ModelCheckpoint callback specifically for the pruned model
pruned_checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(base_path, 'best_pruned_model_weights.h5'),  # Path to save the pruned model file
    save_weights_only=True,            # Save only the weights
    monitor='val_loss',                # Metric to monitor
    save_best_only=True,               # Save only the best model
    verbose=1                          # Log when a model is saved
)

print("New ModelCheckpoint callback for pruned model created.")

New ModelCheckpoint callback for pruned model created.


In [ ]:
# Import the callback needed for pruning
from tensorflow_model_optimization.sparsity.keras import UpdatePruningStep

# Create a list of callbacks for the pruned model
pruning_callbacks = [
    UpdatePruningStep(),
    pruned_checkpoint_callback,  # ModelCheckpoint callback for the pruned model
    early_stopping_callback, # EarlyStopping callback
    reduce_lr # ReduceLROnPlateau callback
]

# Train the pruned model
history_pruned = model_pruning.fit(
    train_generator,
    steps_per_epoch=len(train_images_gray) // train_generator.batch_size,
    epochs=epochs, # Use the same number of epochs defined earlier
    validation_data=val_generator,
    validation_steps=len(val_images_gray) // val_generator.batch_size,
    callbacks=pruning_callbacks # Use the list of pruning and other callbacks
)

print("Pruned model training finished.")

Epoch 1/50
265/265 [==============================] - ETA: 0s - loss: 16.2989 - mean_absolute_error: 3.0240
Epoch 1: val_loss improved from inf to 12.75579, saving model to /content/drive/MyDrive/headposeData/best_pruned_model_weights.h5
265/265 [==============================] - 215s 728ms/step - loss: 16.2989 - mean_absolute_error: 3.0240 - val_loss: 12.7558 - val_mean_absolute_error: 2.7513 - lr: 0.0100
Epoch 2/50
265/265 [==============================] - ETA: 0s - loss: 10.9628 - mean_absolute_error: 2.5044
Epoch 2: val_loss improved from 12.75579 to 7.92731, saving model to /content/drive/MyDrive/headposeData/best_pruned_model_weights.h5
265/265 [==============================] - 196s 740ms/step - loss: 10.9628 - mean_absolute_error: 2.5044 - val_loss: 7.9273 - val_mean_absolute_error: 2.1547 - lr: 0.0100
Epoch 3/50
265/265 [==============================] - ETA: 0s - loss: 8.4763 - mean_absolute_error: 2.2109
Epoch 3: val_loss did not improve from 7.92731
265/265 [==============

KeyboardInterrupt: 

In [ ]:
# Evaluate the model on the test set
test_results = model_pruning.evaluate(
    test_generator,
    steps=len(test_images_gray) // test_generator.batch_size,
    verbose=1
)

# Print the test results
print("\nTest Loss:", test_results[0])
print("Test MAE:", test_results[1])
# print("Test Mean Absolute Error:", test_results[2])

158/158 [==============================] - 13s 81ms/step - loss: 80.4483 - mean_absolute_error: 6.1800

Test Loss: 80.44831848144531
Test MAE: 6.1799845695495605


In [ ]:
from tensorflow_model_optimization.sparsity.keras import strip_pruning

# Strip the pruning mask
stripped_model = strip_pruning(model_pruning)

In [ ]:
stripped_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 64, 64, 1)]       0         
                                                                 
 conv1 (Conv2D)              (None, 32, 32, 32)        288       
                                                                 
 conv1_bn (BatchNormalizatio  (None, 32, 32, 32)       128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 32, 32, 32)        0         
                                                                 
 conv_dw_1 (DepthwiseConv2D)  (None, 32, 32, 32)       288       
                                                                 
 conv_dw_1_bn (BatchNormaliz  (None, 32, 32, 32)       128       
 ation)                                                      

#Convert Pruned model to TFLite

In [ ]:

tf_pruned_model_path=os.path.join(base_path, 'pruned_model_sparse.tflite')
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned_model = converter.convert()
with open(tf_pruned_model_path, 'wb') as f:
  f.write(tflite_pruned_model)

In [ ]:
import numpy as np

def evaluate_lite_model(interpreter, test_images, test_poses):
    """
    Evaluates the TFLite model on the test dataset and calculates Mean Absolute Error (MAE).

    Args:
        interpreter: The TFLite interpreter.
        test_images: The test images (numpy array).
        test_poses: The true test poses (numpy array).

    Returns:
        The Mean Absolute Error (MAE) on the test dataset.
    """
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    all_predictions = []

    for i in range(len(test_images)):
        # Get the input tensor
        input_shape = input_details[0]['shape']
        input_data = np.array(test_images[i], dtype=np.float32) # Ensure input data is float32
        interpreter.set_tensor(input_details[0]['index'], np.expand_dims(input_data, axis=0))

        # Run inference
        interpreter.invoke()

        # Get the output tensor
        output_data = interpreter.get_tensor(output_details[0]['index'])
        all_predictions.append(output_data[0]) # Append the prediction (remove batch dimension)

    all_predictions = np.array(all_predictions)

    # Calculate Mean Absolute Error (MAE)
    mae = np.mean(np.abs(all_predictions - test_poses))

    return mae

# Now, you can call this function after loading the TFLite model:


Here is a function to evaluate the TFLite model on the test dataset using Mean Absolute Error (MAE).

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_pruned_model)
interpreter.allocate_tensors()
test_mae = evaluate_lite_model(interpreter, test_images_gray, test_poses) # Use the grayscale test images and poses
print('TFLite pruned model Test MAE:', test_mae)

TFLite pruned model Test MAE: 17.722235207786333


In [ ]:
#save the baseline model wihtout any optmizations as tfilte
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_baseline = converter.convert()
baselinetf_model_path=os.path.join(base_path, 'baseline_model.tflite')
# Save the baseline TFLite model
with open(baselinetf_model_path, "wb") as f:
    f.write(tflite_model_baseline)


In [ ]:
def get_gzipped_model_size(file):
  # Returns size of gzipped model, in bytes.
  import os
  import zipfile
  import tempfile # Import tempfile here

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file)

In [ ]:
print("Size of gzipped pruned lite  model: %.2f MB" % (get_gzipped_model_size(tf_pruned_model_path)/1e+6))
print("Size of gzipped original lite  model: %.2f MB" % (get_gzipped_model_size(baselinetf_model_path)/1e+6))

Size of gzipped pruned lite  model: 14.42 MB
Size of gzipped original lite  model: 14.42 MB


## Apply Dynamic Range Post-Training Quantization to pruned model

Convert the pruned Keras model to a TensorFlow Lite model with Post-Training Quantization.

What gets quantized:

Weights: int8

Activations: stay float32 in the saved model (converted on-the-fly to float during execution)

Inputs/Outputs: remain float32

In [ ]:
import tensorflow as tf
import os

# Define the file path for the quantized TFLite model
tf_quantized_model_path = os.path.join(base_path, 'pruned_quantized_model.tflite')

# Convert the pruned Keras model to a TFLite model with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model) # Use the stripped model
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

# Save the quantized TFLite model
with open(tf_quantized_model_path, 'wb') as f:
    f.write(tflite_quantized_model)

print(f"Quantized TFLite model saved to: {tf_quantized_model_path}")

Quantized TFLite model saved to: /content/drive/MyDrive/headposeData/pruned_quantized_model.tflite


In [ ]:
import os

# Get the file path of the saved quantized TFLite model
filepath = os.path.join(base_path, 'pruned_quantized_model.tflite')

# Get the file size in bytes
file_size_bytes = os.path.getsize(filepath)

# Convert the size to megabytes (MB) for easier reading
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"Quantized TFLite Model Size: {file_size_mb:.2f} MB")

Quantized TFLite Model Size: 3.88 MB


In [ ]:
print("Size of gzipped quantized lite  model: %.2f MB" % (get_gzipped_model_size(tf_quantized_model_path)/1e+6))

Size of gzipped quantized lite  model: 3.21 MB


#Int 8 quantization

What gets quantized:

Weights: int8

*   Activations: int8
*   Inputs/Outputs: int8

Computation:



*   Runs entirely in int8 (no float ops at all if backend supports it).
*   The representative dataset is used to find min/max ranges for each layer so int8 scaling is accurate.



## Why We Need a Representative Dataset for Full Integer Quantization

When converting a TensorFlow or Keras model to a fully quantized TensorFlow Lite (TFLite) model, **a representative dataset is required** to achieve accurate results. This dataset plays a crucial role during the quantization process.

---

### What Happens During Quantization

Quantization works by mapping floating-point values to a limited integer range (e.g., **int8**: -128 to 127) using a **scale** and **zero-point**:

## Key Properties of a Good Representative Dataset

- **Representative**: It should reflect the data distribution the model will see in production.
- **Small size**: It does not need to be large — typically a few hundred samples is enough.
- **Preprocessed**: Samples must be preprocessed exactly the same way as during training (resize, normalize, etc.).
- **No labels required**: Only inputs are needed, not the ground-truth outputs.


In [ ]:
# Create a generator for a representative dataset
def representative_data_gen():
    # Use a subset of the training data as the representative dataset
    for i in range(min(100, len(train_images_gray))): # Use at most 100 images
        # Add a batch dimension to the image
        yield [np.expand_dims(train_images_gray[i].astype(np.float32), axis=0)]

# Convert the stripped model to a TFLite model with INT8 quantization

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_data_gen


converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
                                            #  tf.lite.OpsSet.SELECT_TF_OPS]
converter_int8.inference_input_type = tf.int8  # Or tf.uint8
#converter_int8.inference_output_type = tf.int8  # Or tf.uint8

tflite_quantized_model_int8 = converter_int8.convert()

# Define the file path for the INT8 quantized TFLite model
tf_quantized_model_int8_path = os.path.join(base_path, 'pruned_quantized_model_int8.tflite')

# Save the INT8 quantized TFLite model
with open(tf_quantized_model_int8_path, 'wb') as f:
    f.write(tflite_quantized_model_int8)

print(f"INT8 Quantized TFLite model saved to: {tf_quantized_model_int8_path}")

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:789: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "


INT8 Quantized TFLite model saved to: /content/drive/MyDrive/headposeData/pruned_quantized_model_int8.tflite


In [ ]:
file_size_bytes = os.path.getsize(tf_quantized_model_int8_path)

# Convert the size to megabytes (MB) for easier reading
file_size_mb = file_size_bytes / (1024 * 1024)

print(f"Quantized TFLite Model Size: {file_size_mb:.2f} MB")

Quantized TFLite Model Size: 4.01 MB


In [ ]:
print("Size of gzipped quantized lite  model: %.2f MB" % (get_gzipped_model_size(tf_quantized_model_int8_path)/1e+6))

Size of gzipped quantized lite  model: 3.27 MB
